# Build a simple RAG pipeline from scratch

**Goal:** by the end of this notebook you will have built, and understood, a complete Retrieval-Augmented Generation (RAG) pipeline that answers questions from your own documents.

**Everything here is free.** No API keys, no paid services:
- **JSON file** holds our source documents
- **SQLite** (built into Python) stores documents, chunks and embeddings
- **sentence-transformers** (runs locally) turns text into embeddings
- **Flan-T5** (a small open model from Google, runs locally) writes the final answer

### What is RAG, in one paragraph?
A language model only knows what it saw in training. It has never seen *your* handbook, policies or notes. RAG fixes this by **looking up** relevant text at question time and **putting it in the prompt**, so the model answers from that text rather than guessing.

### The pipeline we will build

```
 ONE-OFF SETUP ("indexing")                      EVERY QUESTION ("querying")
 ─────────────────────────                       ────────────────────────────
 1. Load documents (JSON)                        5. Embed the question
 2. Split into chunks                            6. Find the most similar chunks
 3. Turn each chunk into an embedding            7. Put those chunks in a prompt
 4. Store chunks + embeddings (SQLite)           8. Model writes the answer
```

Steps 1 to 4 happen once. Steps 5 to 8 happen every time someone asks something. Keeping those two phases separate in your head is the most useful mental model in RAG.

> **How to use this notebook:** run the cells top to bottom (Shift+Enter). Read the explanation above each cell *before* running it. The "Try it yourself" section at the end is where most of the learning happens.

## Python you'll see in this notebook

You don't need to be a Python expert. Here are the handful of ideas the code relies on. Skim this now and come back whenever a line looks unfamiliar.

| You'll see | What it means |
|---|---|
| `my_list = ["a", "b", "c"]` | A **list**: an ordered collection. `my_list[0]` is the first item (counting starts at 0). |
| `my_list[:2]` | A **slice**: the first 2 items. `my_list[-1]` is the last item. `my_list[::-1]` is the list reversed. |
| `person = {"name": "Sam", "age": 30}` | A **dictionary**: labelled values. `person["name"]` gives `"Sam"`. |
| `for item in my_list:` | A **loop**: run the indented code once per item. |
| `for i, item in enumerate(my_list):` | A loop that also gives you the position `i` as it goes. |
| `for a, b in zip(list_1, list_2):` | A loop that walks through two lists side by side. |
| `f"Hello {name}"` | An **f-string**: the `{name}` is replaced by the variable's value. |
| `def add(a, b): return a + b` | A **function**: a reusable block of code. Call it with `add(2, 3)`. |
| `", ".join(["a", "b"])` | Glues a list of strings into one string: `"a, b"`. |
| `!pip install ...` | A `!` at the start of a Colab line runs it as a terminal command, not Python. |
| `"... WHERE id = ?"` with `(5,)` | An **SQL placeholder**: the `?` is filled in safely with the value you pass alongside. |

The next cell lets you try these out. It takes a few seconds and has nothing to do with RAG yet, so just run it and compare the printed results with the table.

**Two other things worth knowing:**
- **Colab basics:** click a cell and press **Shift+Enter** to run it. Run cells in order, top to bottom. A cell with a number like `[3]` beside it has run; `[*]` means it's still running. If something breaks, use **Runtime → Restart session** and run again from the top.
- **NumPy** is a library for working with lists of numbers quickly. A NumPy **array** is like a list, but you can do maths on the whole thing at once. A 2D array (rows and columns) is called a **matrix**.

In [2]:
# Python playground: run this and compare with the table above
#Remember: enumerate() = give each item a number.
#zip() — Pairs items from two lists. Think of zip() as a zip on a jacket. It joins two sides together, one item at a time.
#The difference to remember
# enumerate() = item + its index.
#- zip() = item from one list + item from another list.


fruits = ["apple", "banana", "cherry"]
print("first item:      ", fruits[0])
print("last item:       ", fruits[-1])
print("first two:       ", fruits[:2])
print("reversed:        ", fruits[::-1])

person = {"name": "Sam", "age": 30}
print("dictionary value:", person["name"])

for i, fruit in enumerate(fruits):
    print(f"  item {i} is {fruit}")

for fruit, colour in zip(fruits, ["red", "yellow", "red"]):
    print(f"  a {fruit} is {colour}")

print("joined:          ", ", ".join(fruits))

import numpy as np
numbers = np.array([10, 20, 30])
print("numpy maths:     ", numbers * 2)   # doubles every number at once

first item:       apple
last item:        cherry
first two:        ['apple', 'banana']
reversed:         ['cherry', 'banana', 'apple']
dictionary value: Sam
  item 0 is apple
  item 1 is banana
  item 2 is cherry
  a apple is red
  a banana is yellow
  a cherry is red
joined:           apple, banana, cherry
numpy maths:      [20 40 60]


## Step 0: Setup

We install two libraries. Colab already has `torch` and `numpy`, so this is quick.

- `sentence-transformers`: gives us a pre-trained **embedding model**
- `transformers` + `sentencepiece`: gives us the small **generator model** (Flan-T5)

The first run downloads the models (a few hundred MB), so give it a minute.

In [4]:
%pip install -q sentence-transformers transformers sentencepiece

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import json
import re
import sqlite3

import numpy as np
import torch

from sentence_transformers import SentenceTransformer
from transformers import pipeline

print("Imports OK")

c:\Users\charl\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports OK


## Step 1: Create the knowledge base (a JSON file)

RAG needs something to retrieve from. We will use a small made-up organisation, **Northwind Coding Academy**.

**Why made-up?** If we used a famous company, the model might already "know" the answers from training, and we couldn't tell whether RAG was helping. Because Northwind doesn't exist, any correct answer *must* have come from our retrieved text. That makes it a clean test.

Each document has a `title` and a `text`. In a real project these might be PDFs, web pages or help-centre articles; the principle is the same.

In [ ]:
documents = [
    {
        "title": "Programme overview",
        "text": "Northwind Coding Academy runs a 12-week part-time Data Analytics bootcamp. Live classes take place on Tuesday and Thursday evenings from 6:30pm to 9:00pm UK time. There is also an optional Saturday lab from 10:00am to 12:00pm where students can get help with homework. Each cohort has a maximum of 24 students so that instructors can give individual feedback. New cohorts start in January, May and September."
    },
    {
        "title": "Fees and payment",
        "text": "The bootcamp fee is 3,900 pounds. A non-refundable deposit of 300 pounds secures your place. The remaining balance can be paid in full or in three monthly instalments. Students who pay in full at least 8 weeks before the start date receive a 10 percent early-bird discount. Northwind also offers 20 bursary places per year, each giving 50 percent off the fee to applicants on a low income."
    },
    {
        "title": "Refund and deferral policy",
        "text": "You can receive a full refund, minus the 300 pound deposit, if you cancel at least 14 days before the start date. If you leave during the first two weeks of the course, you will receive a 50 percent refund of the fee. After the end of week two, no refunds are available. Every student may defer to a later cohort once, free of charge, as long as they tell the admissions team before the course starts."
    },
    {
        "title": "Curriculum",
        "text": "Weeks 1 to 3 cover Python and the pandas library for working with data. Weeks 4 to 6 cover SQL, including joins and aggregations, using SQLite. Weeks 7 to 9 cover statistics and data visualisation. Weeks 10 to 12 are a capstone project where each student analyses a real dataset and presents their findings. All tools used on the course are free, including Google Colab and Looker Studio."
    },
    {
        "title": "Admissions",
        "text": "You do not need a university degree to apply. Applicants complete a 45-minute online assessment covering basic maths and logical reasoning. Shortlisted applicants then have a 20-minute interview with a member of the admissions team. Decisions are sent within 5 working days of the interview. All students must be at least 18 years old on the course start date."
    },
    {
        "title": "Careers support",
        "text": "Every student receives three one-to-one career coaching sessions, a CV review and a mock interview. In week 12 there is an employer demo day where students present their capstone projects to hiring managers. Graduates keep access to careers support for 6 months after the course ends. Northwind publishes an outcomes report every year."
    },
    {
        "title": "Equipment and technical requirements",
        "text": "Students need a laptop with at least 8GB of RAM, the Chrome browser and a stable internet connection. All coding is done in the browser, so no software needs to be installed. Northwind has a limited number of loan laptops, 10 per cohort, available for a refundable 50 pound deposit. Loan laptops are allocated on a first-come, first-served basis."
    },
    {
        "title": "Attendance and certificates",
        "text": "Students must attend at least 80 percent of live sessions to receive the course certificate. Recordings of every live class are available for 30 days after the session. If you cannot submit your capstone project on time, you can request a one-week extension by emailing your instructor before the deadline."
    },
]

# Save as a JSON file, exactly as you might receive data in real life
#This creates a real file called knowledge_base.json on your laptop.
#"w" means write.
#json.dump() converts your Python data into JSON and writes it to the file.
#indent=2 makes the JSON easy to read.
with open("knowledge_base.json", "w") as f:
    json.dump(documents, f, indent=2)

# ...Then, you load the JSON back into Python
#json.load() reads the JSON file.
#It converts the JSON data into Python objects.
#docs is now a Python list of dictionaries.
with open("knowledge_base.json") as f:
    docs = json.load(f)

print(f"Loaded {len(docs)} documents:")
for d in docs:
    print(f" - {d['title']} ({len(d['text'].split())} words)")

Loaded 8 documents:
 - Programme overview (66 words)
 - Fees and payment (68 words)
 - Refund and deferral policy (77 words)
 - Curriculum (68 words)
 - Admissions (59 words)
 - Careers support (53 words)
 - Equipment and technical requirements (59 words)
 - Attendance and certificates (50 words)


## Step 2: Create a SQLite database

We could keep everything in Python lists, but a database is closer to how real systems work, and it **persists**: once you've done the expensive work (embedding), you don't have to redo it.

We create two tables:

| Table | Holds | Why |
|---|---|---|
| `documents` | the original documents (id, title, full text) | our source of truth |
| `chunks` | small pieces of each document, plus each piece's **embedding** | this is what we actually search |

The `chunks` table has a `doc_id` linking back to its parent document, so when we retrieve a chunk we can tell the user *where it came from*. Being able to show sources is one of the biggest practical advantages of RAG.

The `embedding` column is a `BLOB` (raw bytes). An embedding is just a list of numbers, and we will pack it into bytes to store it. More on that shortly.

The `DROP TABLE IF EXISTS` lines make this cell safe to re-run.

In [ ]:
conn = sqlite3.connect("rag_demo.db")

conn.executescript("""
DROP TABLE IF EXISTS chunks;
DROP TABLE IF EXISTS documents;

CREATE TABLE documents (
    id    INTEGER PRIMARY KEY,
    title TEXT,
    text  TEXT
);

CREATE TABLE chunks (
    id          INTEGER PRIMARY KEY AUTOINCREMENT,
    doc_id      INTEGER,
    chunk_index INTEGER,
    text        TEXT,
    embedding   BLOB,
    FOREIGN KEY (doc_id) REFERENCES documents(id)
);
""")
#conn = sqlite3.connect("rag_demo.db") here Pythontells SQLite If rag_demo.db already exists, connect to it.
#If it doesn't exist, create a new database file called rag_demo.db.
#above two tables are created in the database. The documents table stores the original documents, and the chunks table stores the text chunks and their embeddings.

for i, d in enumerate(docs, start=1):
    conn.execute(
        "INSERT INTO documents (id, title, text) VALUES (?, ?, ?)", #Each ? is a placeholder.This is called a parameterised query. It's also a safer way to insert data than building SQL statements by joining strings.
        (i, d["title"], d["text"]),
    )
conn.commit()

count = conn.execute("SELECT COUNT(*) FROM documents").fetchone()[0]
print(f"documents table now has {count} rows")

documents table now has 8 rows


In [7]:
rows = conn.execute("SELECT id, title FROM documents").fetchall()

for row in rows:
    print(row)

(1, 'Programme overview')
(2, 'Fees and payment')
(3, 'Refund and deferral policy')
(4, 'Curriculum')
(5, 'Admissions')
(6, 'Careers support')
(7, 'Equipment and technical requirements')
(8, 'Attendance and certificates')


## Step 3: Chunking

**Why not just embed whole documents?**

1. **Precision.** If a document covers five topics and you embed it as one blob, its embedding is a blurry average of all five. A question about one topic matches it weakly. Smaller chunks each have a sharper "meaning".
2. **Prompt size.** The generator can only read so much text. We want to hand it just the relevant bits, not whole documents.

**The trade-off:**
- Chunks too **small** → they lose context ("it costs 300 pounds" but what costs 300 pounds?)
- Chunks too **big** → blurry embeddings and wasted prompt space

**Our strategy:** group whole sentences until a chunk reaches about 60 words, then start a new one. We also repeat the last sentence of each chunk at the start of the next (**overlap**), so an idea that straddles a boundary isn't cut in half.

Real systems use fancier splitters, but this simple sentence-based approach is a perfectly good starting point.

In [ ]:
def chunk_text(text, max_words=60, overlap_sentences=1):
    """Split text into chunks of roughly max_words, never cutting mid-sentence.""" #this is a docstring. It explains what the function does. You can see it in the help() function.
    # Split the text into sentences. The pattern means: "split at any space that
    # comes straight after a full stop, exclamation mark or question mark".
    #re stands for "regular expression". It's a way to describe patterns in text. re.split() splits the text into pieces based on the pattern.
    sentences = re.split(r"(?<=[.!?])\s+", text.strip())

    chunks = []         # finished chunks go here
    current = []        # the sentences in the chunk we're currently building for the next chunk
    word_count = 0      # how many words are in the current chunk

    for sentence in sentences:
        n_words = len(sentence.split())   # splitting on spaces gives us the words

        # If adding this sentence would make the chunk too long, finish this chunk
        if len(current) > 0 and word_count + n_words > max_words:
            chunks.append(" ".join(current))

            # Start the next chunk with the last sentence of this one (the overlap).
            # current[-1:] means "the last 1 item of the list".
            if overlap_sentences > 0:
                current = current[-overlap_sentences:]
            else:
                current = []

            # Recount the words in what we kept
            word_count = 0
            for kept_sentence in current:
                word_count = word_count + len(kept_sentence.split())

        current.append(sentence)
        word_count = word_count + n_words

    # Don't forget the final, partly filled chunk
    if len(current) > 0:
        chunks.append(" ".join(current))
    return chunks


# See it in action on one document
demo_chunks = chunk_text(docs[1]["text"])
print(f"'{docs[1]['title']}' became {len(demo_chunks)} chunks:\n")
for i, c in enumerate(demo_chunks):
    print(f"--- chunk {i} ({len(c.split())} words) ---")
    print(c, "\n")

'Fees and payment' became 2 chunks:

--- chunk 0 (47 words) ---
The bootcamp fee is 3,900 pounds. A non-refundable deposit of 300 pounds secures your place. The remaining balance can be paid in full or in three monthly instalments. Students who pay in full at least 8 weeks before the start date receive a 10 percent early-bird discount. 

--- chunk 1 (40 words) ---
Students who pay in full at least 8 weeks before the start date receive a 10 percent early-bird discount. Northwind also offers 20 bursary places per year, each giving 50 percent off the fee to applicants on a low income. 



**Look at the output above.** Notice that the last sentence of chunk 0 reappears at the start of chunk 1. That's the overlap.

Now we chunk *every* document and save the chunks to the database. We don't have embeddings yet, so that column stays empty for now.

In [9]:
rows = conn.execute("SELECT id, text FROM documents").fetchall()

for doc_id, text in rows:
    for idx, chunk in enumerate(chunk_text(text)):
        conn.execute(
            "INSERT INTO chunks (doc_id, chunk_index, text) VALUES (?, ?, ?)",
            (doc_id, idx, chunk),
        )
conn.commit()

n_chunks = conn.execute("SELECT COUNT(*) FROM chunks").fetchone()[0]
print(f"{len(rows)} documents -> {n_chunks} chunks")

8 documents -> 12 chunks


## Step 4: Embeddings

This is the heart of RAG, so it's worth slowing down.

An **embedding model** reads a piece of text and outputs a list of numbers (a **vector**), here 384 of them. The model has been trained so that **texts with similar meaning get similar vectors**, even when they share no words.

For example, "How much does it cost?" and "What is the tuition fee?" have almost no words in common, but their vectors will be close together. That is what lets RAG find relevant text by *meaning* rather than by exact keyword match.

**How do we measure "close"?** With **cosine similarity**: a score between -1 and 1, where higher means more similar in meaning. If vectors are *normalised* (scaled to length 1), cosine similarity is simply the dot product, which is fast to compute. We pass `normalize_embeddings=True` to arrange that.

Let's load the model and try it on a few sentences before using it on our data.

In [ ]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")

sentences = [
    "How much does the course cost?",        # A
    "What is the tuition fee?",              # B: same meaning as A, different words
    "The cat sat on the sofa.",              # C: unrelated
]

vecs = embedder.encode(sentences, normalize_embeddings=True)#normalisation puts vectors on a consistent length scale so comparisons are easier.

print("Each sentence became a vector of shape:", vecs[0].shape)
print("First 5 numbers of sentence A:", vecs[0][:5].round(3))
print()
print("Similarity A vs B (same meaning):", round(float(vecs[0] @ vecs[1]), 3))
print("Similarity A vs C (unrelated):   ", round(float(vecs[0] @ vecs[2]), 3))

c:\Users\charl\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\charl\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\charl\AppData\Local\Programs\Python\Python312\Lib\site-packages\hu

Each sentence became a vector of shape: (384,)
First 5 numbers of sentence A: [ 0.058  0.007  0.008  0.062 -0.037]

Similarity A vs B (same meaning): 0.693
Similarity A vs C (unrelated):    -0.013


A and B should score much higher than A and C, even though A and B share barely any words. That is the entire trick behind semantic search.

### Embedding all our chunks and storing them

We embed every chunk once and save the vectors in SQLite. Each vector is a NumPy array of `float32` numbers; `.tobytes()` packs it into raw bytes that fit in a `BLOB` column. We'll reverse that with `np.frombuffer` when reading.

This is the expensive step in a big project (thousands of chunks can take a while), which is exactly why we store the results rather than recompute them every time.

In [11]:
chunk_rows = conn.execute("SELECT id, text FROM chunks ORDER BY id").fetchall()
chunk_ids = [r[0] for r in chunk_rows]
chunk_texts = [r[1] for r in chunk_rows]

vectors = embedder.encode(chunk_texts, normalize_embeddings=True, show_progress_bar=True)

for chunk_id, vec in zip(chunk_ids, vectors):   # walk through ids and vectors side by side
    # Make sure the numbers are 32-bit decimals (so every vector is stored the same way)
    vec_32 = vec.astype(np.float32)

    # Pack the array of numbers into raw bytes, which is what a BLOB column holds
    vec_bytes = vec_32.tobytes()

    # Save the bytes into this chunk's row
    conn.execute(
        "UPDATE chunks SET embedding = ? WHERE id = ?",
        (vec_bytes, chunk_id),
    )
conn.commit()

print(f"Stored {len(vectors)} embeddings, each with {vectors.shape[1]} numbers")

Batches: 100%|██████████| 1/1 [00:00<00:00,  2.70it/s]

Stored 12 embeddings, each with 384 numbers


## Step 5: Retrieval

Now the part that happens at question time. To retrieve:

1. **Embed the question** using the *same* model (this is essential: you can only compare vectors made by the same model)
2. **Compare** it to every chunk's vector (one dot product each)
3. **Return the top *k*** highest-scoring chunks

First we load all embeddings from SQLite into one NumPy **matrix**, with one row per chunk (and 384 columns, one per number in the vector). We also keep a separate list, `meta`, holding each chunk's text and title in the *same order*, so row 5 of the matrix always belongs to `meta[5]`.

Then comes the one slightly mathematical idea: the **dot product**. To compare two vectors, multiply the numbers in matching positions and add up the results:

```
question vector:  [1, 2, 3]
chunk vector:     [4, 5, 6]
dot product:      1×4 + 2×5 + 3×6 = 32
```

Because our vectors are normalised, this single number *is* the cosine similarity: bigger means closer in meaning. Doing it for every chunk gives one score per chunk.

Finally we need the **top k**. NumPy's `argsort` doesn't sort the scores themselves; it tells you the *positions* you'd visit to read them from lowest to highest:

```
scores:            [0.2, 0.9, 0.5]
argsort:           [0, 2, 1]     (position 0 is lowest, then position 2, then position 1 is highest)
reversed:          [1, 2, 0]     (now best first)
first 2 of those:  [1, 2]        (the positions of the 2 best scores)
```

The code below does exactly this, one small step at a time.

> **Is scanning every chunk too slow?** For thousands of chunks, no, it takes milliseconds. Vector databases (Pinecone, Chroma, FAISS and so on) exist for millions of chunks. For learning, and for many small real projects, a NumPy matrix is all you need. Don't reach for a vector database until you have a reason.

In [ ]:
def load_index():
    """Read all chunks and their embeddings from SQLite into memory."""
    # SQL: for each chunk, also fetch the title of the document it belongs to.
    # JOIN means "match each chunk's doc_id with the document that has that id".
    rows = conn.execute("""
        SELECT c.id, d.title, c.text, c.embedding
        FROM chunks c
        JOIN documents d ON d.id = c.doc_id
        ORDER BY c.id
    """).fetchall()

    meta = []       # will hold each chunk's id, title and text
    vectors = []    # will hold each chunk's embedding

    for row in rows:
        chunk_id, title, text, embedding_bytes = row

        # Keep the readable details in one dictionary
        meta.append({"chunk_id": chunk_id, "title": title, "text": text})

        # Turn the stored bytes back into an array of 384 numbers
        vectors.append(np.frombuffer(embedding_bytes, dtype=np.float32))

    # Stack the 384-number arrays on top of each other to make one matrix
    matrix = np.vstack(vectors)
    return meta, matrix


meta, matrix = load_index()
print("Embedding matrix shape (chunks x dimensions):", matrix.shape)


def retrieve(question, k=3):
    """Return the k chunks most similar in meaning to the question."""
    # 1. Turn the question into a vector, using the SAME model as for the chunks
    q_vec = embedder.encode([question], normalize_embeddings=True)[0]

    # 2. Score every chunk against the question with the dot product.
    #    `matrix` has one row per chunk, so this gives one score per chunk.
    scores = np.dot(matrix, q_vec)

    # 3. Get the chunk positions ordered from lowest score to highest score
    positions_low_to_high = np.argsort(scores)

    # 4. Reverse that, so the best-scoring chunk comes first
    positions_best_first = positions_low_to_high[::-1]

    # 5. Keep only the first k positions
    top_positions = positions_best_first[:k]

    # 6. Build a readable result for each of the top chunks
    results = []
    for position in top_positions:
        chunk = meta[position]
        results.append({
            "chunk_id": chunk["chunk_id"],
            "title": chunk["title"],
            "text": chunk["text"],
            "score": float(scores[position]),
        })
    return results

In [ ]:
question = "Can I get my money back if I quit in week 3?"

for hit in retrieve(question, k=3):
    print(f"score {hit['score']:.3f} | from: {hit['title']}")
    print(f"   {hit['text']}\n")

**Check the output.** The top result should come from "Refund and deferral policy". Notice the question never used the word "refund", and the document never said "money back" or "quit". The embedding model bridged that gap.

This is a good moment to appreciate that **retrieval is working independently of any language model**. We haven't generated a single word yet. In practice, most RAG problems are retrieval problems, so it's worth inspecting retrieval output like this whenever answers go wrong.

## Step 6: The generator

Now we need a model that can read the retrieved chunks and write an answer. We use **Flan-T5**, a small open instruction-following model that runs on Colab's free CPU.

**Be realistic:** it's tiny compared with models like Claude or GPT. Its answers will be short and sometimes clumsy. That's fine for learning, because the point is to see the *pipeline* work. In a real project you would swap this one function for a call to a stronger model, and nothing else in the pipeline would change. This is another nice property of RAG: the parts are separate.

First, an experiment. Let's ask the model a question about Northwind **without** giving it any context.

In [ ]:
generator = pipeline("text2text-generation", model="google/flan-t5-base")

def generate(prompt, max_new_tokens=100):
    out = generator(prompt, max_new_tokens=max_new_tokens, do_sample=False)
    return out[0]["generated_text"]

# No retrieval, no context: the model has to guess
print(generate("How much does the Northwind Coding Academy bootcamp cost?"))

The model has never heard of Northwind, so whatever it said was a guess. Without RAG, a model will confidently produce *something*. This is the problem RAG exists to solve.

## Step 7: Putting it all together

The final piece is the **prompt**. We give the model:
1. An instruction: answer using *only* the context, and admit when the answer isn't there
2. The retrieved chunks as context
3. The question

That "say I don't know" instruction matters. Without it, models tend to fill gaps with plausible-sounding inventions, which defeats the point of grounding the answer in your documents.

> **A note on limits:** Flan-T5 can only read about 512 tokens (roughly 350 words). With 3 chunks of around 60 words that is comfortably within budget. If you raise `k` or chunk size a lot, the end of the prompt may be silently cut off. Larger models have much bigger limits.

In [ ]:
def build_prompt(question, chunks):
    # Collect the text of each retrieved chunk
    chunk_texts = []
    for chunk in chunks:
        chunk_texts.append(chunk["text"])

    # Join them into one block of context, with a blank line between chunks
    context = "\n\n".join(chunk_texts)

    # Fill the context and question into the prompt template
    return f"""Answer the question using only the context below. If the answer is not in the context, say "I don't know".

Context:
{context}

Question: {question}
Answer:"""


def ask(question, k=3, show_sources=True):
    """The whole RAG pipeline: retrieve, build prompt, generate."""
    # Retrieve: find the k most relevant chunks
    chunks = retrieve(question, k=k)

    # Augment: put those chunks and the question into a prompt
    prompt = build_prompt(question, chunks)

    # Generate: let the model write the answer
    answer = generate(prompt)

    print(f"Q: {question}")
    print(f"A: {answer}")

    if show_sources:
        # Build a short label like "Fees and payment (0.62)" for each source chunk
        labels = []
        for chunk in chunks:
            labels.append(f"{chunk['title']} ({chunk['score']:.2f})")
        print("   Sources:", ", ".join(labels))
    print()


ask("How much does the bootcamp cost?")
ask("Can I get my money back if I quit in week 3?")
ask("Do I need a degree to apply?")
ask("What laptop do I need?")
ask("Which days are classes on?")

**You've just built RAG.** Compare these answers with the guess from Step 6: the model now gives correct, specific answers it could not have known on its own.

Notice also the **Sources** line. For every answer we can see exactly which documents it was based on, and the similarity scores. That makes answers checkable, which is a major reason people choose RAG over fine-tuning.

### What happens when the answer *isn't* in the documents?
A good RAG system should say it doesn't know rather than make something up. Let's test that.

In [ ]:
ask("Does Northwind offer a Rust programming course?")
ask("Who is the CEO of Northwind Coding Academy?")
ask("What is the capital of France?")

Results here can be mixed with a small model. Sometimes it says "I don't know", sometimes it grabs something loosely related. Look at the **scores** too: irrelevant questions tend to produce lower similarity scores than relevant ones.

This is an important lesson. **A RAG system always retrieves *something*** (the top-k nearest chunks, however weak). It's up to you to handle bad matches, for example by setting a minimum score threshold, or by using a stronger model that follows the "I don't know" instruction reliably. You'll try a threshold in the exercises.

## Step 8: A first taste of evaluation

How do you know your pipeline is any good? So far we've eyeballed a few questions, which doesn't scale and can fool you.

The simplest useful check targets **retrieval** only: for a set of questions where *you know which document should contain the answer*, does that document appear in the top-k results? This is called a **hit rate**.

This is a preview of the next step in learning RAG (proper evaluation), but even this tiny version is valuable.

In [ ]:
# (question, title of the document that should be retrieved)
test_set = [
    ("How much does the bootcamp cost?",                "Fees and payment"),
    ("Is there a discount for paying early?",           "Fees and payment"),
    ("What happens if I want to cancel?",               "Refund and deferral policy"),
    ("Can I postpone to a later cohort?",               "Refund and deferral policy"),
    ("Is a degree required?",                           "Admissions"),
    ("How long does the application process take?",     "Admissions"),
    ("What will I learn in the first month?",           "Curriculum"),
    ("Will you help me find a job afterwards?",         "Careers support"),
    ("Can I borrow a laptop?",                          "Equipment and technical requirements"),
    ("What if I miss some classes?",                    "Attendance and certificates"),
    ("When are the live sessions?",                     "Programme overview"),
]

def hit_rate(k=3):
    hits = 0    # how many questions retrieved the right document

    for question, expected_title in test_set:
        # Retrieve the top k chunks and note which documents they came from
        retrieved_chunks = retrieve(question, k=k)
        retrieved_titles = []
        for chunk in retrieved_chunks:
            retrieved_titles.append(chunk["title"])

        # Was the document we wanted among them?
        if expected_title in retrieved_titles:
            hits = hits + 1
        else:
            print(f"MISS: '{question}' expected '{expected_title}', got {retrieved_titles}")

    percentage = hits / len(test_set)
    print(f"k={k}: {hits}/{len(test_set)} correct ({percentage:.0%})")

hit_rate(k=1)
hit_rate(k=3)

Compare `k=1` with `k=3`. Retrieving more chunks gives the right document more chances to appear, but it also puts more (possibly irrelevant) text in the prompt. That tension between recall and noise comes up constantly in RAG.

If any questions missed, look at *why*. That's where you start learning how your pipeline really behaves.

---

## Try it yourself

This is where it sticks. Change one thing at a time and see what happens.

1. **Ask your own questions.** Try ones that need two documents ("What does it cost and what's the refund policy?"). Does the pipeline cope?
2. **Change chunk size.** Re-run Steps 2 to 5 with `max_words=25`, then `max_words=150`. (Rerun from the SQLite cell, which resets the tables.) How do the hit rate and answer quality change?
3. **Change `overlap_sentences`** to 0. Do any answers break?
4. **Add a score threshold.** In `ask`, if the best score is below, say, 0.3, print "I don't know" without calling the generator. Find a threshold that rejects the Rust and France questions but keeps the real ones. (Look at the scores to choose.)
5. **Add your own documents.** Edit `knowledge_base.json` (or the `documents` list) with text you care about, such as notes or a policy, and rebuild.
6. **Break it deliberately.** Write a question using words that appear in the documents but means something different. Does retrieval get fooled?
7. **Swap the generator.** Try `google/flan-t5-large` in the `pipeline(...)` line. Are the answers better? Slower?

---

## Recap

| Stage | What we did | Key idea |
|---|---|---|
| Load | JSON file → SQLite | Persist your data so you don't redo work |
| Chunk | Split into ~60-word pieces with overlap | Small chunks = sharper matching |
| Embed | Text → 384 numbers | Similar meaning = similar vectors |
| Retrieve | Cosine similarity, top-k | Search by meaning, not keywords |
| Generate | Chunks + question → prompt → answer | Ground the model in your text |
| Evaluate | Hit rate on a test set | Measure, don't guess |

### Where next
- **Step 2: evaluation in depth.** Build a larger test set, measure retrieval and answer quality separately, and learn to spot *why* things fail.
- **Step 3: agents.** Wrap `retrieve()` as a **tool** and let a model decide when to call it, rewrite its query and try again if the results look weak. The `retrieve` function you wrote here is exactly what you would hand to that agent.